# L03 – Lösningar
Kör först setup-cellerna nedan (de motsvarar avsnitt 1 i lektionen).

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import fetch_california_housing, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

def set_seed(seed: int = 42) -> None:
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

device = "cuda" if torch.cuda.is_available() else "cpu"

X, y = fetch_california_housing(return_X_y=True)
X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=0.25, random_state=42)
scaler = StandardScaler().fit(X_train)
X_train_s, X_val_s = scaler.transform(X_train), scaler.transform(X_val)

def to_tensors(X, y):
    return (torch.tensor(X, dtype=torch.float32, device=device),
            torch.tensor(y, dtype=torch.float32, device=device).unsqueeze(1))

Xtr, ytr = to_tensors(X_train_s, y_train)
Xva, yva = to_tensors(X_val_s, y_val)
rmse_lin = root_mean_squared_error(y_val, LinearRegression().fit(X_train_s, y_train).predict(X_val_s))

### Övning 1

In [ ]:
logits = torch.tensor([[1.0, 2.0, 0.5, 0.0],
                       [3.0, -1.0, 0.0, 1.0],
                       [0.0, 0.0, 0.0, 0.0]])
targets = torch.tensor([1, 0, 3])

probs = torch.softmax(logits, dim=1)
my_ce = -torch.log(probs[torch.arange(len(targets)), targets]).mean()
assert torch.isclose(my_ce, nn.CrossEntropyLoss()(logits, targets))
print("Rätt!", my_ce.item())

# Tredje exemplet: alla logits lika → softmax ger 1/4 till varje klass → loss = -ln(1/4) = ln 4
print("Exempel 3:", -torch.log(probs[2, 3]).item(), " ln 4 =", np.log(4))

Med lika logits är modellen helt osäker: den ger samma sannolikhet $1/K$ till alla klasser. Loss blir då $\ln K$. Det är också den loss man **förväntar sig i början** av träningen med $K$ klasser, en bra sanity check som vi återkommer till i L09.

### Övning 2

In [ ]:
set_seed(42)
model = nn.Linear(8, 1).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
loss_fn = nn.MSELoss()
batch_size = 64

# Med lr=1e-2 blir slutet för brusigt (stegen studsar runt minimum), med 3e-3 räcker 20 epoker.
for epoch in range(20):
    perm = torch.randperm(len(Xtr), device=device)
    for i in range(0, len(Xtr), batch_size):
        idx = perm[i:i + batch_size]
        optimizer.zero_grad()
        loss = loss_fn(model(Xtr[idx]), ytr[idx])
        loss.backward()
        optimizer.step()

with torch.no_grad():
    val_rmse = loss_fn(model(Xva), yva).sqrt().item()
assert val_rmse < rmse_lin * 1.01, f"{val_rmse:.4f} jämfört med {rmse_lin:.4f}"
print(f"Rätt! val RMSE {val_rmse:.4f} (sklearn {rmse_lin:.4f})")

### Övning 3

In [ ]:
Xb, yb = load_breast_cancer(return_X_y=True)
Xb_tr, Xb_va, yb_tr, yb_va = train_test_split(Xb, yb, test_size=0.25, stratify=yb, random_state=42)

sc = StandardScaler().fit(Xb_tr)
Xbt, ybt = to_tensors(sc.transform(Xb_tr), yb_tr)   # y blir float32 och (N, 1), som BCE vill ha
Xbv, ybv = to_tensors(sc.transform(Xb_va), yb_va)

set_seed(42)
clf = nn.Linear(30, 1).to(device)
optimizer = torch.optim.Adam(clf.parameters(), lr=1e-2)
bce = nn.BCEWithLogitsLoss()

for epoch in range(200):
    optimizer.zero_grad()
    loss = bce(clf(Xbt), ybt)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    preds = (clf(Xbv) > 0).float()     # logit > 0  ⇔  sigmoid(logit) > 0.5
    val_acc = (preds == ybv).float().mean().item()
assert val_acc > 0.95, val_acc
print(f"Rätt! val accuracy {val_acc:.3f}")

### Övning 4 – förslag på svar
1. `CrossEntropyLoss` gör softmax (egentligen log-softmax) internt, på ett numeriskt stabilt sätt. Om du lägger till en egen softmax görs den två gånger. Gradienterna blir då små och träningen sämre.
2. Sänk learning rate (till exempel med en faktor 10). Kontrollera också att indata är normaliserad och inte innehåller `nan`.
3. Annars ser modellen batcherna i samma ordning varje epok. Uppdateringarna blir systematiskt snedvridna (till exempel om datan är sorterad efter klass), och bruset som hjälper generaliseringen försvinner.